## Agentic agent experiment

This experiment explores how agentic agent decides which tools to use

Hypothesis: agentic approach reliability can be improved by adding explicit measurement.

Some ideas:
- We should explicitly define what sufficient means and how to evaluate if the information is sufficient

Future work
- Can we optimise the agent nodes/process? At the moment, it's taking time to produce an answer.

Sufficient_metric e.g. comprehensive

Questions: what type of questions cause this?

In [100]:
%load_ext autoreload
%load_ext dotenv
%dotenv -o ../tests/.env.test
%aimport redbox

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Testing agentic prompt

In [111]:
from redbox.models.settings import Settings, ElasticLocalSettings
from pathlib import Path
from dotenv import load_dotenv, dotenv_values
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers.string import StrOutputParser
from langchain_core.prompts.chat import ChatPromptTemplate
import pandas as pd

ENV = Settings(
    minio_host="localhost", 
    object_store="minio", 
    unstructured_host="localhost",
    worker_ingest_min_chunk_size=1_000,
    worker_ingest_max_chunk_size=10_000,
    worker_ingest_largest_chunk_size=300_000,
    elastic=ElasticLocalSettings(host="localhost"),
)


LLM = AzureChatOpenAI(
    api_key=ENV.embedding_openai_api_key,
    azure_endpoint=ENV.embedding_azure_openai_endpoint,
    model='gpt-4o'
)
TEST_INDEX = "rag-test-index"

In [123]:
judging_criteria = [
    {
        "criteria": "Relevance",
        "definition": "",
        "scores": [
            {
                "value": 1,
                "definition": "The information is largely irrelevant to the question.",
                "indicators": [
                    {
                        "indicator": "Most of the key points and data do not relate to the problem at hand."
                    },
                    {
                        "indicator": "The information is mostly extraneous or off-topic."
                    }
                ]
            },
            {
                "value": 2,
                "definition": "The information has some relevance but misses many key aspects of the question.",
                "indicators": [
                    {
                        "indicator": "Only a few points or data are related to the problem."
                    },
                    {
                        "indicator": "There is significant irrelevant information."
                    }
                ]
            },
            {
                "value": 3,
                "definition": "The information is moderately relevant but not comprehensive.",
                "indicators": [
                    {
                        "indicator": "A fair number of points and data are related to the question, but some key aspects are missing."
                    },
                    {
                        "indicator": "Some irrelevant information is present."
                    }
                ]
            },
            {
                "value": 4,
                "definition": "The information is mostly relevant with minor gaps.",
                "indicators": [
                    {
                        "indicator": "Most key points and data are directly related to the question."
                    },
                    {
                        "indicator": "Very little irrelevant information is present."
                    }
                ]
            },
            {
                "value": 5,
                "definition": "The information directly and comprehensively addresses the question.",
                "indicators": [
                    {
                        "indicator": "All key points and data are directly related to the problem at hand."
                    },
                    {
                        "indicator": "No extraneous or irrelevant information is present."
                    }
                ]
            }
        ]
    }, 
    {
        "criteria": "Completeness",
        "definition": "",
        "scores": [
            {
                "value": 1,
                "definition": "The information is highly incomplete.",
                "indicators": [
                    {
                        "indicator": "Few parts of the question are answered."
                    },
                    {
                        "indicator": "Major elements and data points are missing."
                    }
                ]
            },
            {
                "value": 2,
                "definition": "The information is somewhat incomplete",
                "indicators": [
                    {
                        "indicator": "Some parts of the question are answered, but significant elements are missing."
                    },
                    {
                        "indicator": "Many gaps in the data."
                    }
                ]
            },
            {
                "value": 3,
                "definition": "The information is moderately complete.",
                "indicators": [
                    {
                        "indicator": "Many parts of the question are answered, but some important elements are missing."
                    },
                    {
                        "indicator": "Some gaps in the data."
                    }
                ]
            },
            {
                "value": 4,
                "definition": "The information is mostly complete with minor gaps",
                "indicators": [
                    {
                        "indicator": "Most parts of the question are answered."
                    },
                    {
                        "indicator": "A few minor elements or data points are missing."
                    }
                ]
            },
            {
                "value": 5,
                "definition": "The information fully covers all aspects of the question.",
                "indicators": [
                    {
                        "indicator": "Every part of the question is answered with no missing elements."
                    },
                    {
                        "indicator": "The provided data and points are exhaustive in scope."
                    }
                ]
            },
        ]
    },
    {
        "criteria": "Consistency",
        "definition": "",
        "scores": [
            {
                "value": 1,
                "definition": "The information is highly inconsistent with many contradictions.",
                "indicators": [
                    {
                        "indicator": "Significant discrepancies and conflicting data across sources."
                    },
                    {
                        "indicator": "The information does not form a unified narrative."
                    }
                ]
            },
            {
                "value": 2,
                "definition": "The information has some inconsistencies and contradictions.",
                "indicators": [
                    {
                        "indicator": "Several discrepancies and conflicting data points."
                    },
                    {
                        "indicator": "The information forms a somewhat disjointed narrative."
                    }
                ]
            },
            {
                "value": 3,
                "definition": "The information is moderately consistent with some minor contradictions.",
                "indicators": [
                    {
                        "indicator": "A few discrepancies and minor conflicting data points."
                    },
                    {
                        "indicator": "The information forms a mostly coherent narrative."
                    }
                ]
            },
            {
                "value": 4,
                "definition": "The information is mostly consistent with minor issues.",
                "indicators": [
                    {
                        "indicator": "Very few discrepancies or conflicting data points."
                    },
                    {
                        "indicator": "The information forms a coherent narrative."
                    }
                ]
            },
            {
                "value": 5,
                "definition": "The information is consistent across different documents and tool call results with no contradictions.",
                "indicators": [
                    {
                        "indicator": "All data points and information are in agreement across sources."
                    },
                    {
                        "indicator": "There are no discrepancies or conflicting information."
                    }
                ]
            },
        ]
    },
    # {
    #     "criteria": "",
    #     "definition": "",
    #     "scores": [
    #         {
    #             "value": 1,
    #             "definition": "",
    #             "indicators": [
    #                 {
    #                     "indicator": ""
    #                 },
    #                 {
    #                     "indicator": ""
    #                 }
    #             ]
    #         },
    #         {
    #             "value": 2,
    #             "definition": "",
    #             "indicators": [
    #                 {
    #                     "indicator": ""
    #                 },
    #                 {
    #                     "indicator": ""
    #                 }
    #             ]
    #         },
    #         {
    #             "value": 3,
    #             "definition": "",
    #             "indicators": [
    #                 {
    #                     "indicator": ""
    #                 },
    #                 {
    #                     "indicator": ""
    #                 }
    #             ]
    #         },
    #         {
    #             "value": 4,
    #             "definition": "",
    #             "indicators": [
    #                 {
    #                     "indicator": ""
    #                 },
    #                 {
    #                     "indicator": ""
    #                 }
    #             ]
    #         },
    #         {
    #             "value": 5,
    #             "definition": "",
    #             "indicators": [
    #                 {
    #                     "indicator": ""
    #                 },
    #                 {
    #                     "indicator": ""
    #                 }
    #             ]
    #         },
    #     ]
    # }
]

In [124]:
agentic_system_prompt = (
    "You are an advanced problem-solving assistant. Your primary goal is to carefully "
    "analyse and work through complex questions or problems. You will receive a collection "
    "of documents (all at once, without any information about their order or iteration) and "
    "a list of tool calls that have already been made (also without order or iteration "
    "information). Based on this data, you are expected to think critically about how to "
    "proceed.\n"
    "\n"
    "Objective:\n"
    "1. Examine the available documents and tool calls:\n"
    "- Evaluate whether the current information is sufficient to answer the question.\n"
    "- Consider the success or failure of previous tool calls based on the data they returned.\n"
    "- Hypothesise whether new tool calls might bring more valuable information.\n"
    "\n"
    "2. Decide how to proceed:\n"
    "- If additional tool calls are likely to yield useful information, make those calls.\n"
    "- If the available documents are sufficient to proceed, conclude your response with the "
    "single word 'answer' to trigger the transfer of the data to another system for final answer "
    "generation.\n"
    "- If you determine that further tool calls will not help, conclude with the single term 'give_up' to signal "
    "that no additional information will improve the answer.\n"
    "\n"
    "Your role is to think deeply before taking any action. Carefully weigh whether new "
    "information is necessary or helpful. Only take action (call tools, 'give_up', or trigger an "
    "'answer') after thorough evaluation of the current documents and tool calls."
)

agentic_system_prompt_new = (
    "You are an advanced problem-solving assistant. Your primary goal is to carefully "
    "analyse and work through complex questions or problems. You will receive a collection "
    "of documents (all at once, without any information about their order or iteration) and "
    "a list of tool calls that have already been made (also without order or iteration "
    "information). Based on this data, you are expected to think critically about how to "
    "proceed.\n"
    "\n"
    "Objective:\n"
    "1. Examine the available documents and tool calls:\n"
    "- Evaluate whether the current information is sufficient to answer the question using {judging_criteria}\n"
    "\n"
    "2. Decide how to proceed:\n"
    "- If additional tool calls are likely to yield useful information, make those calls.\n"
    "- If the evaluation score is higher than 4, conclude your response with the "
    "single word 'answer' to trigger the transfer of the data to another system for final answer "
    "generation.\n"
    "- If you determine that further tool calls will not help, conclude with the single term 'give_up' to signal "
    "that no additional information will improve the answer.\n"
    "\n"
    "Only take action (call tools, 'give_up', or trigger an "
    "'answer') after thorough evaluation of the current documents and tool calls. Return evaluation score."
)

agentic_retrival_prompt = (
    "The following context and previous actions are provided to assist you. \n\n"
    "Previous tool calls: \n\n <ToolCalls> \n\n  {tool_calls} </ToolCalls> \n\n "
    "Document snippets: \n\n <Documents> \n\n {formatted_documents} </Documents> \n\n "
    "User question: \n\n {question}"
)

In [15]:
query = "@gadget Who is hello kitty?"

In [125]:
tool_calls = ""
formatted_documents = 'Hello kitty is a cat.'
documents = []
tool_call = []
analysis = []
decision = []
action = []
next_call = []
responses = []
new_agent = []

for i in range(6):

    if i<=2:
        chain = ChatPromptTemplate(
            [
                agentic_system_prompt_new,
                agentic_retrival_prompt
            ]
        ) | LLM | StrOutputParser()

        response = chain.invoke({ "question": query, 'tool_calls': tool_calls, 'formatted_documents': formatted_documents, 'judging_criteria': judging_criteria})
        new_agent += ['Yes']
    else:
        chain = ChatPromptTemplate(
            [
                agentic_system_prompt,
                agentic_retrival_prompt
            ]
        ) | LLM | StrOutputParser()
        response = chain.invoke({ "question": query, 'tool_calls': tool_calls, 'formatted_documents': formatted_documents})
        new_agent += ['No']
    
    responses += [response]
    
# ex.to_csv('Users/saisakulchernbumroong/Documents/agentic_reliability.csv')


INFO:httpx:HTTP Request: POST https://oai-i-dot-ai-playground-sweden.openai.azure.com//openai/deployments/gpt-4o/chat/completions?api-version=2024-02-01 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://oai-i-dot-ai-playground-sweden.openai.azure.com//openai/deployments/gpt-4o/chat/completions?api-version=2024-02-01 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://oai-i-dot-ai-playground-sweden.openai.azure.com//openai/deployments/gpt-4o/chat/completions?api-version=2024-02-01 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://oai-i-dot-ai-playground-sweden.openai.azure.com//openai/deployments/gpt-4o/chat/completions?api-version=2024-02-01 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://oai-i-dot-ai-playground-sweden.openai.azure.com//openai/deployments/gpt-4o/chat/completions?api-version=2024-02-01 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://oai-i-dot-ai-playground-sweden.openai.azure.com//openai/deployments/gpt-4o/chat/completions?api-version=202

In [95]:
response

'### Evaluation of Available Information\n\n#### Relevance Criteria:\n1. **Relevance Definition**: The information should directly and comprehensively address the question "Who is Hello Kitty?"\n2. **Scores and Indicators**:\n\n   - **Value 1**: The information is largely irrelevant to the question.\n     - Indicators: Most key points and data do not relate to the problem at hand; the information is mostly extraneous or off-topic.\n   - **Value 2**: The information has some relevance but misses many key aspects of the question.\n     - Indicators: Only a few points or data are related to the problem; there is significant irrelevant information.\n   - **Value 3**: The information is moderately relevant but not comprehensive.\n     - Indicators: A fair number of points and data are related to the question, but some key aspects are missing; some irrelevant information is present.\n   - **Value 4**: The information is mostly relevant with minor gaps.\n     - Indicators: Most key points and

In [46]:
import re
def extract_tool_and_query(text):
    # Define the regular expression patterns to match the "tool" and "query" fields
    tool_pattern = re.compile(r'"tool":\s*"([^"]+)"')
    query_pattern = re.compile(r'"query":\s*"([^"]+)"')

    # Search for the patterns in the given text
    tool_match = tool_pattern.search(text)
    query_match = query_pattern.search(text)

    # Extract the matched groups if they exist
    tool = tool_match.group(1) if tool_match else None
    query = query_match.group(1) if query_match else None

    return [f'Tool call: {tool}, query: {query}']

In [126]:
pd.DataFrame({'response': responses, 'Using_judging_scheme': new_agent}).to_csv('/Users/saisakulchernbumroong/Documents/agentic_reliability_with_score1.csv')

In [ ]:
# documents = re.findall(r'Available Document Snippet:\n(.*)', response)
# tool_calls = re.findall(r'Previous Tool Call:\n(.*)', response)
# analysis = re.findall(r'\*\*Sufficiency of the Document\*\*:(.*)', response)
# decision = re.findall(r'Decision:\n(.*)', response)
# action = re.findall(r'Action:\n(.*)', response)

